## Setup

In [ ]:
import asyncio
import base64
import json
import os
from pprint import pprint

from dotenv import load_dotenv

from agent_base.core.messages import Message
from agent_base.core.provider import RetryPolicy
from agent_base.core.types import ContentBlockType, DocumentContent, ImageContent, Role, SourceType, TextContent, ToolResultContent
from agent_base.providers.any_llm import AnyLLMConfig, AnyLLMMessageFormatter, AnyLLMProvider
from agent_base.tools.tool_types import ToolSchema

load_dotenv('../../../.env')

# any-llm model ids are colon-form "provider:model", e.g. "openai:gpt-4o-mini",
# "anthropic:claude-haiku-4-5". Keys come from the provider's env var
# (OPENAI_API_KEY, ANTHROPIC_API_KEY, ...) unless AnyLLMConfig.api_key is set.
PRIMARY_MODEL = os.getenv('ANY_LLM_TEST_MODEL_PRIMARY')
SECONDARY_MODEL = os.getenv('ANY_LLM_TEST_MODEL_SECONDARY')
REASONING_MODEL = os.getenv('ANY_LLM_TEST_MODEL_REASONING', PRIMARY_MODEL or '')

assert PRIMARY_MODEL, 'Set ANY_LLM_TEST_MODEL_PRIMARY in .env or the notebook environment.'

formatter = AnyLLMMessageFormatter()
any_llm_provider = AnyLLMProvider(
    formatter=formatter,
    retry_policy=RetryPolicy(max_retries=1, base_delay=0.1),
)


class ListSink:
    """Minimal DeltaSink: collects typed StreamDeltas for assertions."""

    def __init__(self):
        self.deltas = []
        self.metas = []

    def emit(self, delta):
        self.deltas.append(delta)

    def emit_meta(self, body):
        self.metas.append(body)


weather_schema = ToolSchema(
    name='get_weather',
    description='Get current weather for a city. Always call this tool when asked about weather.',
    input_schema={
        'type': 'object',
        'properties': {'city': {'type': 'string'}},
        'required': ['city'],
    },
)

## Provider Round Trip Tests

Canonical `Message` -> format -> `any_llm.acompletion` -> parse -> `ProviderTurn` (the canonical assistant `Message` rides on `.message`; `was_cancelled`, `partial_error` and provider-private `stream_bookkeeping` ride alongside).

**Deliberate design note:** this provider exposes *no* provider-specific features (server tools, citations, prompt caching, skills) as first-class config. Anything provider-specific rides the two escape hatches on `AnyLLMConfig` — `api_kwargs` (merged verbatim into the request, applied last) and `client_args` (forwarded to the provider client constructor). See the Escape Hatch sections below and `DESIGN.md`.

### Basic - Primary Model

In [ ]:
canonical_message = Message.user('Reply with exactly: PONG')

res = await any_llm_provider.generate(
    system_prompt='You are a test bot. Follow instructions exactly.',
    messages=[canonical_message],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=64),
)

pprint(res.message.to_dict())
assert res.was_cancelled is False
assert res.partial_error is None
assert res.message.role == Role.ASSISTANT
assert res.message.provider == 'any_llm'
assert res.message.model
assert res.message.usage is not None
assert res.message.usage.input_tokens > 0
assert res.message.usage.output_tokens > 0
assert any(block.content_block_type == ContentBlockType.TEXT for block in res.message.content)

### Basic - Secondary Model (Optional)

In [ ]:
if not SECONDARY_MODEL:
    print('Skipping secondary-model round trip: ANY_LLM_TEST_MODEL_SECONDARY not set.')
else:
    res2 = await any_llm_provider.generate(
        system_prompt='You are a test bot. Follow instructions exactly.',
        messages=[canonical_message],
        tool_schemas=[],
        model=SECONDARY_MODEL,
        llm_config=AnyLLMConfig(max_tokens=64),
    )
    pprint(res2.message.to_dict())
    assert res2.message.role == Role.ASSISTANT
    assert res2.message.provider == 'any_llm'
    assert res2.message.usage is not None
    assert any(block.content_block_type == ContentBlockType.TEXT for block in res2.message.content)

### Streaming - Text Deltas

Incremental `TextDelta`s stream through the sink; the final `ProviderTurn.message` is rebuilt from the same chunks, so the concatenated deltas must equal the final text.

In [ ]:
sink = ListSink()
stream_res = await any_llm_provider.generate_stream(
    system_prompt='You are a test bot.',
    messages=[Message.user('Count from 1 to 5 as digits separated by spaces.')],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=64),
    sink=sink,
)

text_deltas = [d for d in sink.deltas if getattr(d, 'type', '') == 'text']
streamed_text = ''.join(d.text for d in text_deltas)
print(streamed_text)

assert text_deltas, 'expected incremental TextDeltas'
assert stream_res.was_cancelled is False
assert stream_res.message.role == Role.ASSISTANT
final_text = ''.join(
    block.text for block in stream_res.message.content
    if block.content_block_type == ContentBlockType.TEXT
)
assert final_text == streamed_text
# include_usage is an OpenAI-convention passthrough, not an any-llm guarantee;
# usage-in-stream is provider-dependent (holds for the default openai model).
assert stream_res.message.usage is not None

### Streaming - Cancellation

The provider checks `cancellation_event` on every chunk; a set event returns promptly with `was_cancelled=True` and the partial message built from what already streamed.

In [ ]:
cancel_event = asyncio.Event()
cancel_event.set()  # pre-cancelled: the stream must bail on the first check

cancel_sink = ListSink()
cancel_res = await any_llm_provider.generate_stream(
    system_prompt=None,
    messages=[Message.user('Write a long poem about the sea.')],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=512),
    sink=cancel_sink,
    cancellation_event=cancel_event,
)

print(f'was_cancelled={cancel_res.was_cancelled}, deltas seen: {len(cancel_sink.deltas)}')
assert cancel_res.was_cancelled is True
assert cancel_res.message.role == Role.ASSISTANT

### Client Tools

In [ ]:
tool_prompt = Message.user("What's the weather in Paris?")

tool_res = await any_llm_provider.generate(
    system_prompt='Always use the get_weather tool for weather questions. Never answer directly.',
    messages=[tool_prompt],
    tool_schemas=[weather_schema],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=128),
)

pprint(tool_res.message.to_dict())
assert tool_res.message.stop_reason == 'tool_use'
tool_block = next(
    block for block in tool_res.message.content
    if block.content_block_type == ContentBlockType.TOOL_USE
)
assert tool_block.tool_name == 'get_weather'
assert tool_block.tool_id
assert isinstance(tool_block.tool_input, dict)

In [ ]:
# Tool result follow-up: complete the loop with a role="tool" round trip.
tool_result_message = Message(
    role=Role.USER,
    content=[
        ToolResultContent(
            tool_name=tool_block.tool_name,
            tool_id=tool_block.tool_id,
            tool_result='Sunny, 22C',
        ),
        TextContent(text='Summarize that briefly.'),
    ],
)

tool_followup = await any_llm_provider.generate(
    system_prompt='Use tool outputs when they are provided.',
    messages=[tool_prompt, tool_res.message, tool_result_message],
    tool_schemas=[weather_schema],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=128),
)

pprint(tool_followup.message.to_dict())
combined_text = ' '.join(
    block.text for block in tool_followup.message.content
    if block.content_block_type == ContentBlockType.TEXT
).lower()
assert combined_text
assert 'sunny' in combined_text or '22' in combined_text or 'paris' in combined_text

### Streaming - Tool Call Deltas

Tool calls are emitted **buffered** — one terminal `ToolCallDelta` with the full accumulated `arguments_json` per call (never incremental fragments).

In [ ]:
sink = ListSink()
stream_tool_res = await any_llm_provider.generate_stream(
    system_prompt='Always use the get_weather tool for weather questions. Never answer directly.',
    messages=[Message.user("What's the weather in Berlin?")],
    tool_schemas=[weather_schema],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=256),
    sink=sink,
)

tool_call_deltas = [d for d in sink.deltas if getattr(d, 'type', '') == 'tool_call']
for d in tool_call_deltas:
    print(d.tool_name, d.arguments_json)

assert tool_call_deltas, 'expected at least one ToolCallDelta'
assert all(d.is_final for d in tool_call_deltas)
assert all(json.loads(d.arguments_json) for d in tool_call_deltas)
assert stream_tool_res.message.stop_reason == 'tool_use'

### Image Input

In [ ]:
img_b64 = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAHggJ/PchI7wAAAABJRU5ErkJggg=='

image_res = await any_llm_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        ImageContent(media_type='image/png', source_type=SourceType.BASE64, data=img_b64),
        TextContent(text='What color is this image?'),
    ])],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=128),
)

pprint(image_res.message.to_dict())
assert any(block.content_block_type == ContentBlockType.TEXT for block in image_res.message.content)

### Plain Text Document Input

In [ ]:
doc_text = 'The speed of light is approximately 299,792,458 meters per second.'
doc_b64 = base64.b64encode(doc_text.encode()).decode()

doc_res = await any_llm_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type='text/plain', source_type=SourceType.BASE64, data=doc_b64),
        TextContent(text='What speed is mentioned in this document?'),
    ])],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=128),
)

pprint(doc_res.message.to_dict())
doc_text_out = ' '.join(
    block.text for block in doc_res.message.content
    if block.content_block_type == ContentBlockType.TEXT
)
assert doc_text_out
assert '299' in doc_text_out or 'speed of light' in doc_text_out.lower()

### Reasoning / Thinking (Optional)

`reasoning_effort` is the ONE first-class reasoning knob (any-llm harmonises it across providers). Reasoning comes back as `message.reasoning` -> a leading `ThinkingContent` block; in streams it arrives as `delta.reasoning` -> `ThinkingDelta`.

In [ ]:
if os.getenv('ANY_LLM_ENABLE_REASONING_TEST') != '1' or not REASONING_MODEL:
    print('Skipping reasoning test: enable with ANY_LLM_ENABLE_REASONING_TEST=1.')
else:
    reasoning_res = await any_llm_provider.generate(
        system_prompt=None,
        messages=[Message.user('What is 25 * 37?')],
        tool_schemas=[],
        model=REASONING_MODEL,
        llm_config=AnyLLMConfig(max_tokens=2048, reasoning_effort='low'),
    )
    pprint(reasoning_res.message.to_dict())
    assert any(
        block.content_block_type == ContentBlockType.TEXT
        for block in reasoning_res.message.content
    )
    thinking_blocks = [
        block for block in reasoning_res.message.content
        if block.content_block_type == ContentBlockType.THINKING
    ]
    print(f'Thinking blocks returned: {len(thinking_blocks)}')
    if reasoning_res.message.usage:
        print(f'thinking_tokens: {reasoning_res.message.usage.thinking_tokens}')

### Escape Hatch - Per-Request api_kwargs

`AnyLLMConfig.api_kwargs` is merged verbatim into the `acompletion` call, **applied last** (it wins over provider-built params). This is THE mechanism for anything not first-class: generic OpenAI-style params (`temperature`, `response_format`, `tool_choice`, `parallel_tool_calls`, ...) and truly provider-specific kwargs (e.g. Mistral's `safe_prompt`, an Anthropic `thinking` dict) alike.

In [ ]:
res_hatch = await any_llm_provider.generate(
    system_prompt='Return a JSON object with a single key "answer" whose value is the number 42.',
    messages=[Message.user('Respond with the JSON object only.')],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(
        max_tokens=64,
        api_kwargs={'temperature': 0, 'response_format': {'type': 'json_object'}},
    ),
)

raw = ''.join(
    block.text for block in res_hatch.message.content
    if block.content_block_type == ContentBlockType.TEXT
)
print(raw)
payload = json.loads(raw)
assert payload.get('answer') == 42

### Escape Hatch - client_args

`AnyLLMConfig.client_args` reaches the any-llm provider *client constructor* (not the request) — the sanctioned home for timeouts, proxies and default headers. any-llm has no top-level `timeout` param.

In [ ]:
res_client = await any_llm_provider.generate(
    system_prompt=None,
    messages=[Message.user('Reply with exactly: OK')],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=AnyLLMConfig(max_tokens=16, client_args={'timeout': 60}),
)

print(''.join(
    block.text for block in res_client.message.content
    if block.content_block_type == ContentBlockType.TEXT
))
assert res_client.message.usage.output_tokens > 0

### Error Classification

`classify_error` maps any failure from the any-llm call onto the 8-member canonical `ErrorCode` — the loop never sniffs native exceptions. (Works whether or not `ANY_LLM_UNIFIED_EXCEPTIONS` is enabled.)

In [ ]:
from agent_base.core.errors import ErrorCode
from agent_base.core.provider import ProviderError

try:
    await any_llm_provider.generate(
        system_prompt=None,
        messages=[Message.user('hi')],
        tool_schemas=[],
        model='openai:definitely-not-a-real-model-xyz',
        llm_config=AnyLLMConfig(max_tokens=16),
    )
    raise AssertionError('expected the bogus model to fail')
except AssertionError:
    raise
except Exception as exc:
    classified = exc if isinstance(exc, ProviderError) else any_llm_provider.classify_error(exc)
    print(f'code={classified.code}, native_code={classified.native_code}, retriable={classified.retriable}')
    assert isinstance(classified.code, ErrorCode)
    assert classified.retriable is False  # model-not-found is not retriable